In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from scipy import stats


plt.rcParams.update({
    "font.size": 8,
    "axes.linewidth": 0.8,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
})


def load_results(csv_file):
    """Load earthquake catalog and log-likelihood results."""
    data = pd.read_csv(csv_file)
    mask = data["step"].isin(["train", "val", "test"])
    return {
        "ll_t": data.loc[mask, "logli_t"].values,
        "ll_s": data.loc[mask, "logli_s"].values,
        "train_num": int((data["step"] == "train").sum()),
        "val_num": int((data["step"] == "val").sum()),
        "test_num": int((data["step"] == "test").sum()),
    }


def plot_regression(x, y, dataset_id, likelihood_type):
    """Plot regression between ST-NKF and ETAS log-likelihoods."""
    xy = np.vstack([x, y])
    density = stats.gaussian_kde(xy)(xy)
    order = density.argsort()
    x, y, density = x[order], y[order], density[order]

    slope, intercept = np.polyfit(x, y, 1)
    r2 = stats.pearsonr(x, y).statistic ** 2

    fig, ax = plt.subplots(figsize=(3.5, 3), dpi=300)

    sc = ax.scatter(
        x, y, c=density * 100, s=10,
        cmap="RdBu_r", alpha=0.8, edgecolors="none"
    )

    fig.colorbar(sc, ax=ax, shrink=0.8, pad=0.02,
                 aspect=20, label="Density")

    ax.plot(
        x,
        slope * x + intercept,
        color="red",
        lw=1.5,
        label=fr"Regression, $R^2$={r2:.4f}",
    )

    lim_min = min(x.min(), y.min())
    lim_max = max(x.max(), y.max())

    ax.plot(
        [lim_min, lim_max],
        [lim_min, lim_max],
        "--",
        color="#444444",
        lw=1.5,
        label="1:1 Line",
    )

    if likelihood_type == "temporal":
        ax.set_xlabel(r"$\mathcal{L}_{temporal,\ ST\!-\!NKF}$")
        ax.set_ylabel(r"$\mathcal{L}_{temporal,\ ETAS}$")
    else:
        ax.set_xlabel(r"$\mathcal{L}_{spatial,\ ST\!-\!NKF}$")
        ax.set_ylabel(r"$\mathcal{L}_{spatial,\ ETAS}$")

    ax.grid(True, ls="--", lw=0.3, alpha=0.5)
    ax.legend(frameon=False, fontsize=8)

    ax.tick_params(axis="both", rotation=45, length=3)
    ax.xaxis.set_major_locator(MaxNLocator())
    ax.yaxis.set_major_locator(MaxNLocator())

    fig.tight_layout()

    outdir = Path("fig")
    outdir.mkdir(exist_ok=True)
    fig.savefig(outdir / f"regression-{dataset_id}_{likelihood_type}.png",
                dpi=300, bbox_inches="tight")
    plt.close(fig)



In [2]:
datasets = [
    "ComCat", "SaltonSea", "SanJac", "WHITE",
    "SCEDC_20", "SCEDC_25", "SCEDC_30",
]

for dataset in datasets:
    etas = load_results(f"./csv/{dataset}_empirical_empirical_empirical.csv")
    nkf = load_results(f"./csv/{dataset}_neural_neural_neural.csv")

    start = etas["train_num"] + etas["val_num"]

    print(
        f"{dataset}: train={etas['train_num']}, "
        f"val={etas['val_num']}, test={etas['test_num']}"
    )

    plot_regression(
        nkf["ll_t"][start:],
        etas["ll_t"][start:],
        dataset,
        "temporal",
    )

    plot_regression(
        nkf["ll_s"][start:],
        etas["ll_s"][start:],
        dataset,
        "spatial",
    )

ComCat: train=40701, val=14741, test=21888
SaltonSea: train=36639, val=2683, test=4104
SanJac: train=11770, val=3449, test=4400
WHITE: train=18548, val=13812, test=24080
SCEDC_20: train=81433, val=23484, test=13071
SCEDC_25: train=25257, val=9089, test=5000
SCEDC_30: train=6815, val=3135, test=1898
